# Modern Baseline Benchmarking for Cross-Script Writer Verification

## Research Question

How competitive is the current batch-level adversarial writer-verification model when compared with stronger modern visual representations under the exact same writer-disjoint verification protocol?

## Motivation

The current batch-alternating model consistently improved writer verification and cross-script verification across multiple training seeds. However, its absolute performance cannot be interpreted as state of the art without comparison against strong modern representation baselines evaluated under the same data split and pair protocol.

This notebook therefore performs controlled benchmarking rather than introducing a new method.

## Benchmarking Principles

- Use the existing writer-disjoint QUWI split.
- Use the exact existing validation verification pairs.
- Preserve the same cosine-similarity verification protocol.
- Report ROC-AUC and interpolated EER.
- Report overall, within-script, and cross-script performance.
- Do not use the official test set for model or hyperparameter selection.
- Start with frozen modern representations before introducing adaptation.
- Use development writers only for any learned adaptation such as LDA.
- Compare all methods under the same evaluation protocol.

## Current Reference

The main reference model is the batch-level alternating adversarial model from the previous experiments.

The purpose of this notebook is not to claim state-of-the-art performance immediately, but to determine whether the existing verifier is competitive enough to support the next reliability and uncertainty-aware research phase.

In [1]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [2]:
SPLIT_SEED = 42
EMBEDDING_BATCH_SIZE = 16

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

NOTEBOOK17_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "multiseed_adversarial_robustness"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modern_baseline_benchmarking"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_df = pd.read_csv(
    SPLIT_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .reset_index(drop=True)
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

multiseed_writer_df = pd.read_csv(
    NOTEBOOK17_REPORT_DIR
    / "multiseed_writer_summary.csv"
)

multiseed_cross_df = pd.read_csv(
    NOTEBOOK17_REPORT_DIR
    / "multiseed_cross_script_summary.csv"
)

optional_packages = {
    package: (
        importlib.util.find_spec(
            package
        )
        is not None
    )
    for package in [
        "timm",
        "transformers",
    ]
}

TRAIN_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Device:",
    TRAIN_DEVICE,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "Torchvision:",
    torchvision.__version__,
)

print()

print(
    "Development writers:",
    development_df[
        "writer"
    ].nunique(),
)

print(
    "Development images:",
    len(
        development_df
    ),
)

print(
    "Validation writers:",
    validation_df[
        "writer"
    ].nunique(),
)

print(
    "Validation images:",
    len(
        validation_df
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print()

print(
    "Current batch-alt mean AUC:",
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean(),
)

print(
    "Current batch-alt mean cross-script AUC:",
    multiseed_cross_df[
        "batch_cross_auc"
    ].mean(),
)

print()

print(
    "Optional packages:",
    optional_packages,
)

Project root: C:\Users\com\Documents\handwriting-cross-script-research
Device: cuda
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130

Development writers: 226
Development images: 904
Validation writers: 56
Validation images: 224
Validation pairs: 18816

Current batch-alt mean AUC: 0.7773429715952037
Current batch-alt mean cross-script AUC: 0.7460860196351268

Optional packages: {'timm': False, 'transformers': False}


In [3]:
VIT_WEIGHTS = (
    torchvision.models
    .ViT_B_16_Weights
    .IMAGENET1K_SWAG_E2E_V1
)

VIT_MODEL_NAME = (
    "vit_b_16_swag_e2e_frozen"
)

VIT_INPUT_SIZE = 384

print(
    "Baseline:",
    VIT_MODEL_NAME,
)

print(
    "Input size:",
    VIT_INPUT_SIZE,
)

print(
    "ImageNet top-1:",
    VIT_WEIGHTS.meta[
        "_metrics"
    ][
        "ImageNet-1K"
    ][
        "acc@1"
    ],
)

print(
    "Parameters:",
    VIT_WEIGHTS.meta[
        "num_params"
    ],
)

print(
    "Weights file size (MB):",
    VIT_WEIGHTS.meta[
        "_file_size"
    ],
)

Baseline: vit_b_16_swag_e2e_frozen
Input size: 384
ImageNet top-1: 85.304
Parameters: 86859496
Weights file size (MB): 331.398


In [4]:
vit_model = (
    torchvision.models.vit_b_16(
        weights=VIT_WEIGHTS
    )
)

vit_model.heads = (
    torch.nn.Identity()
)

vit_model = vit_model.to(
    TRAIN_DEVICE
)

vit_model.eval()

for parameter in (
    vit_model.parameters()
):
    parameter.requires_grad = False

dummy_image = torch.zeros(
    1,
    3,
    VIT_INPUT_SIZE,
    VIT_INPUT_SIZE,
    device=TRAIN_DEVICE,
)

with torch.no_grad():
    dummy_embedding = (
        vit_model(
            dummy_image
        )
    )

print(
    "Model device:",
    next(
        vit_model.parameters()
    ).device,
)

print(
    "Embedding shape:",
    dummy_embedding.shape,
)

print(
    "Embedding dimension:",
    dummy_embedding.shape[
        1
    ],
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in (
            vit_model.parameters()
        )
        if parameter.requires_grad
    ),
)

Model device: cuda:0
Embedding shape: torch.Size([1, 768])
Embedding dimension: 768
Trainable parameters: 0


In [5]:
from handwriting_cross_script_research.dataset import QUWIDataset

VIT_BATCH_SIZE = 8

vit_preprocess = (
    VIT_WEIGHTS.transforms()
)

validation_dataset = QUWIDataset(
    metadata=validation_df,
    image_dir=IMAGE_DIR,
)

validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

sanity_batch = next(
    iter(
        validation_loader
    )
)

sanity_images = sanity_batch[
    "image"
]

sanity_processed = (
    vit_preprocess(
        sanity_images
    )
)

print(
    "Raw batch shape:",
    sanity_images.shape,
)

print(
    "Raw value range:",
    float(
        sanity_images.min()
    ),
    float(
        sanity_images.max()
    ),
)

print(
    "Processed batch shape:",
    sanity_processed.shape,
)

print(
    "Processed value range:",
    float(
        sanity_processed.min()
    ),
    float(
        sanity_processed.max()
    ),
)

print(
    "Validation batches:",
    len(
        validation_loader
    ),
)

Raw batch shape: torch.Size([8, 3, 384, 384])
Raw value range: 0.25882354378700256 1.0
Processed batch shape: torch.Size([8, 3, 384, 384])
Processed value range: -0.9876701831817627 2.640000104904175
Validation batches: 28


In [6]:
def extract_vit_embeddings(
    model,
    loader,
):
    embeddings = []
    filenames = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            images = (
                batch[
                    "image"
                ]
                .to(
                    TRAIN_DEVICE
                )
            )

            processed_images = (
                vit_preprocess(
                    images
                )
            )

            batch_embeddings = (
                model(
                    processed_images
                )
            )

            batch_embeddings = (
                torch.nn.functional.normalize(
                    batch_embeddings,
                    p=2,
                    dim=1,
                )
            )

            embeddings.append(
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        filenames,
    )


vit_validation_embeddings, vit_validation_filenames = (
    extract_vit_embeddings(
        vit_model,
        validation_loader,
    )
)

print(
    "Validation embeddings:",
    vit_validation_embeddings.shape,
)

print(
    "Validation filenames:",
    len(
        vit_validation_filenames
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Minimum embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).min(),
)

print(
    "Maximum embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).max(),
)

Validation embeddings: (224, 768)
Validation filenames: 224
Mean embedding norm: 1.0
Minimum embedding norm: 0.9999999
Maximum embedding norm: 1.0000001


In [7]:
def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = thresholds[
            nearest_index
        ]

        return float(
            eer
        ), float(
            threshold
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return float(
        eer
    ), float(
        threshold
    )


vit_embedding_map = dict(
    zip(
        vit_validation_filenames,
        vit_validation_embeddings,
    )
)

vit_embeddings_a = np.stack(
    [
        vit_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

vit_embeddings_b = np.stack(
    [
        vit_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

vit_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

vit_scores = np.sum(
    vit_embeddings_a
    * vit_embeddings_b,
    axis=1,
)

vit_overall_auc = roc_auc_score(
    vit_pair_labels,
    vit_scores,
)

vit_overall_eer, vit_eer_threshold = (
    calculate_interpolated_eer(
        vit_pair_labels,
        vit_scores,
    )
)

current_batch_mean_auc = (
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean()
)

print(
    "Frozen ViT overall AUC:",
    vit_overall_auc,
)

print(
    "Frozen ViT EER (%):",
    100.0
    * vit_overall_eer,
)

print(
    "Frozen ViT EER threshold:",
    vit_eer_threshold,
)

print()

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print(
    "ViT AUC difference:",
    vit_overall_auc
    - current_batch_mean_auc,
)

Frozen ViT overall AUC: 0.6713422490208204
Frozen ViT EER (%): 38.39285714285714
Frozen ViT EER threshold: 0.8898820241815165

Current batch-alt mean AUC: 0.7773429715952037
ViT AUC difference: -0.10600072257438331


In [8]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


vit_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

vit_pair_df[
    "score"
] = vit_scores

vit_pair_df[
    "page_a"
] = vit_pair_df[
    "filename_a"
].map(
    filename_page_id
)

vit_pair_df[
    "page_b"
] = vit_pair_df[
    "filename_b"
].map(
    filename_page_id
)

vit_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(
                page_a
            ),
            int(
                page_b
            ),
        )
    ]
    for page_a, page_b in zip(
        vit_pair_df[
            "page_a"
        ],
        vit_pair_df[
            "page_b"
        ],
    )
]

vit_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = (
        vit_pair_df[
            vit_pair_df[
                "condition"
            ] == condition
        ]
    )

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    vit_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )


vit_condition_df = pd.DataFrame(
    vit_condition_rows
)

vit_within_macro_auc = (
    vit_condition_df[
        vit_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

vit_cross_macro_auc = (
    vit_condition_df[
        vit_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

current_batch_cross_auc = (
    multiseed_cross_df[
        "batch_cross_auc"
    ].mean()
)

print(
    vit_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT within-script macro AUC:",
    vit_within_macro_auc,
)

print(
    "Frozen ViT cross-script macro AUC:",
    vit_cross_macro_auc,
)

print()

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print(
    "ViT cross-script AUC difference:",
    vit_cross_macro_auc
    - current_batch_cross_auc,
)

              condition      auc      eer
   arabic_variable_same 0.726467 0.332468
  english_variable_same 0.752690 0.333117
cross_variable_variable 0.664561 0.397078
    cross_variable_same 0.596104 0.428571
    cross_same_variable 0.663248 0.392857
        cross_same_same 0.667341 0.375000

Frozen ViT within-script macro AUC: 0.7395785018552876
Frozen ViT cross-script macro AUC: 0.6478135146103896

Current batch-alt mean cross-script AUC: 0.7460860196351268
ViT cross-script AUC difference: -0.09827250502473717


In [9]:
development_dataset = QUWIDataset(
    metadata=development_df,
    image_dir=IMAGE_DIR,
)

development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

vit_development_embeddings, vit_development_filenames = (
    extract_vit_embeddings(
        vit_model,
        development_loader,
    )
)

development_writer_map = dict(
    zip(
        development_df[
            "filename"
        ],
        development_df[
            "writer"
        ],
    )
)

development_writer_labels = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in (
            vit_development_filenames
        )
    ]
)

print(
    "Development embeddings:",
    vit_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        vit_development_filenames
    ),
)

print(
    "Development writers:",
    np.unique(
        development_writer_labels
    ).size,
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            development_writer_labels,
            return_counts=True,
        )[
            1
        ]
    ).tolist(),
)

Development embeddings: (904, 768)
Development filenames: 904
Development writers: 226
Samples per writer: [4]


In [10]:
writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

writer_lda.fit(
    vit_development_embeddings,
    development_writer_labels,
)

lda_development_embeddings = (
    writer_lda.transform(
        vit_development_embeddings
    )
)

lda_validation_embeddings = (
    writer_lda.transform(
        vit_validation_embeddings
    )
)

lda_development_embeddings = (
    lda_development_embeddings
    / np.linalg.norm(
        lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

lda_validation_embeddings = (
    lda_validation_embeddings
    / np.linalg.norm(
        lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

lda_embedding_map = dict(
    zip(
        vit_validation_filenames,
        lda_validation_embeddings,
    )
)

lda_embeddings_a = np.stack(
    [
        lda_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

lda_embeddings_b = np.stack(
    [
        lda_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

lda_scores = np.sum(
    lda_embeddings_a
    * lda_embeddings_b,
    axis=1,
)

lda_overall_auc = roc_auc_score(
    vit_pair_labels,
    lda_scores,
)

lda_overall_eer, lda_eer_threshold = (
    calculate_interpolated_eer(
        vit_pair_labels,
        lda_scores,
    )
)

lda_pair_df = (
    vit_pair_df
    .copy()
)

lda_pair_df[
    "score"
] = lda_scores

lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = (
        lda_pair_df[
            lda_pair_df[
                "condition"
            ] == condition
        ]
    )

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    lda_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )


lda_condition_df = pd.DataFrame(
    lda_condition_rows
)

lda_within_macro_auc = (
    lda_condition_df[
        lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

lda_cross_macro_auc = (
    lda_condition_df[
        lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    lda_validation_embeddings.shape[
        1
    ],
)

print()

print(
    "Frozen ViT overall AUC:",
    vit_overall_auc,
)

print(
    "ViT + writer-LDA overall AUC:",
    lda_overall_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "ViT + writer-LDA EER (%):",
    100.0
    * lda_overall_eer,
)

print()

print(
    lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT cross-script macro AUC:",
    vit_cross_macro_auc,
)

print(
    "ViT + writer-LDA cross-script macro AUC:",
    lda_cross_macro_auc,
)

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print()

print(
    "LDA vs batch-alt overall difference:",
    lda_overall_auc
    - current_batch_mean_auc,
)

print(
    "LDA vs batch-alt cross-script difference:",
    lda_cross_macro_auc
    - current_batch_cross_auc,
)

LDA embedding dimension: 225

Frozen ViT overall AUC: 0.6713422490208204
ViT + writer-LDA overall AUC: 0.703562570861678
Current batch-alt mean AUC: 0.7773429715952037

ViT + writer-LDA EER (%): 35.0

              condition      auc      eer
   arabic_variable_same 0.769724 0.296753
  english_variable_same 0.830740 0.232792
cross_variable_variable 0.651380 0.388312
    cross_variable_same 0.644191 0.379545
    cross_same_variable 0.640068 0.392857
        cross_same_same 0.687100 0.375000

Frozen ViT cross-script macro AUC: 0.6478135146103896
ViT + writer-LDA cross-script macro AUC: 0.6556847170686456
Current batch-alt mean cross-script AUC: 0.7460860196351268

LDA vs batch-alt overall difference: -0.07378040073352576
LDA vs batch-alt cross-script difference: -0.09040130256648116


In [11]:
EFFICIENTNET_WEIGHTS = torchvision.models.EfficientNet_V2_L_Weights.IMAGENET1K_V1
EFFICIENTNET_MODEL_NAME = "efficientnet_v2_l_frozen"
EFFICIENTNET_BATCH_SIZE = 4

efficientnet_preprocess = EFFICIENTNET_WEIGHTS.transforms()

efficientnet_model = torchvision.models.efficientnet_v2_l(
    weights=EFFICIENTNET_WEIGHTS
)

efficientnet_model.classifier = torch.nn.Identity()
efficientnet_model = efficientnet_model.to(TRAIN_DEVICE)
efficientnet_model.eval()

for parameter in efficientnet_model.parameters():
    parameter.requires_grad = False

torch.cuda.empty_cache()

dummy_image = torch.zeros(
    1,
    3,
    384,
    384,
    device=TRAIN_DEVICE,
)

with torch.no_grad():
    dummy_processed = efficientnet_preprocess(dummy_image)
    dummy_embedding = efficientnet_model(dummy_processed)

print("Baseline:", EFFICIENTNET_MODEL_NAME)
print("Processed input shape:", dummy_processed.shape)
print("Embedding shape:", dummy_embedding.shape)
print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in efficientnet_model.parameters()
        if parameter.requires_grad
    ),
)
print(
    "ImageNet top-1:",
    EFFICIENTNET_WEIGHTS.meta["_metrics"]["ImageNet-1K"]["acc@1"],
)
print(
    "GPU memory allocated (GB):",
    torch.cuda.memory_allocated() / 1024**3,
)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_l-59c71312.pth" to C:\Users\com/.cache\torch\hub\checkpoints\efficientnet_v2_l-59c71312.pth


100.0%


Baseline: efficientnet_v2_l_frozen
Processed input shape: torch.Size([1, 3, 480, 480])
Embedding shape: torch.Size([1, 1280])
Trainable parameters: 0
ImageNet top-1: 85.808
GPU memory allocated (GB): 0.7889304161071777


In [12]:
efficientnet_validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=EFFICIENTNET_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_efficientnet_embeddings(model, loader):
    embeddings = []
    filenames = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(TRAIN_DEVICE)

            processed_images = efficientnet_preprocess(images)
            batch_embeddings = model(processed_images)

            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

            embeddings.append(
                batch_embeddings.detach().cpu().numpy()
            )

            filenames.extend(
                list(batch["filename"])
            )

    return (
        np.concatenate(embeddings, axis=0),
        filenames,
    )


efficientnet_validation_embeddings, efficientnet_validation_filenames = (
    extract_efficientnet_embeddings(
        efficientnet_model,
        efficientnet_validation_loader,
    )
)

print(
    "Validation embeddings:",
    efficientnet_validation_embeddings.shape,
)
print(
    "Validation filenames:",
    len(efficientnet_validation_filenames),
)
print(
    "Mean embedding norm:",
    np.linalg.norm(
        efficientnet_validation_embeddings,
        axis=1,
    ).mean(),
)
print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated() / 1024**3,
)

Validation embeddings: (224, 1280)
Validation filenames: 224
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 0.956967830657959


In [13]:
def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = thresholds[
            nearest_index
        ]

        return float(
            eer
        ), float(
            threshold
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return float(
        eer
    ), float(
        threshold
    )


efficientnet_embedding_map = dict(
    zip(
        efficientnet_validation_filenames,
        efficientnet_validation_embeddings,
    )
)

efficientnet_embeddings_a = np.stack(
    [
        efficientnet_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

efficientnet_embeddings_b = np.stack(
    [
        efficientnet_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

efficientnet_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

efficientnet_scores = np.sum(
    efficientnet_embeddings_a
    * efficientnet_embeddings_b,
    axis=1,
)

efficientnet_overall_auc = roc_auc_score(
    efficientnet_pair_labels,
    efficientnet_scores,
)

efficientnet_overall_eer, efficientnet_eer_threshold = (
    calculate_interpolated_eer(
        efficientnet_pair_labels,
        efficientnet_scores,
    )
)

current_batch_mean_auc = (
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean()
)

print(
    "Frozen EfficientNetV2-L overall AUC:",
    efficientnet_overall_auc,
)

print(
    "Frozen EfficientNetV2-L EER (%):",
    100.0
    * efficientnet_overall_eer,
)

print(
    "Frozen EfficientNetV2-L EER threshold:",
    efficientnet_eer_threshold,
)

print()

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print(
    "EfficientNetV2-L AUC difference:",
    efficientnet_overall_auc
    - current_batch_mean_auc,
)

CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


efficientnet_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

efficientnet_pair_df[
    "score"
] = efficientnet_scores

efficientnet_pair_df[
    "page_a"
] = efficientnet_pair_df[
    "filename_a"
].map(
    filename_page_id
)

efficientnet_pair_df[
    "page_b"
] = efficientnet_pair_df[
    "filename_b"
].map(
    filename_page_id
)

efficientnet_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(
                page_a
            ),
            int(
                page_b
            ),
        )
    ]
    for page_a, page_b in zip(
        efficientnet_pair_df[
            "page_a"
        ],
        efficientnet_pair_df[
            "page_b"
        ],
    )
]

efficientnet_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = (
        efficientnet_pair_df[
            efficientnet_pair_df[
                "condition"
            ] == condition
        ]
    )

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    efficientnet_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )


efficientnet_condition_df = pd.DataFrame(
    efficientnet_condition_rows
)

efficientnet_within_macro_auc = (
    efficientnet_condition_df[
        efficientnet_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

efficientnet_cross_macro_auc = (
    efficientnet_condition_df[
        efficientnet_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

current_batch_cross_auc = (
    multiseed_cross_df[
        "batch_cross_auc"
    ].mean()
)

print()

print(
    efficientnet_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen EfficientNetV2-L within-script macro AUC:",
    efficientnet_within_macro_auc,
)

print(
    "Frozen EfficientNetV2-L cross-script macro AUC:",
    efficientnet_cross_macro_auc,
)

print()

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print(
    "EfficientNetV2-L cross-script AUC difference:",
    efficientnet_cross_macro_auc
    - current_batch_cross_auc,
)

Frozen EfficientNetV2-L overall AUC: 0.6733993313234385
Frozen EfficientNetV2-L EER (%): 37.5
Frozen EfficientNetV2-L EER threshold: 0.894299004226923

Current batch-alt mean AUC: 0.7773429715952037
EfficientNetV2-L AUC difference: -0.10394364027176517

              condition      auc      eer
   arabic_variable_same 0.766083 0.301948
  english_variable_same 0.778160 0.321429
cross_variable_variable 0.674896 0.357143
    cross_variable_same 0.580554 0.483766
    cross_same_variable 0.659879 0.398052
        cross_same_same 0.671144 0.375000

Frozen EfficientNetV2-L within-script macro AUC: 0.7721214053803339
Frozen EfficientNetV2-L cross-script macro AUC: 0.64661844851577

Current batch-alt mean cross-script AUC: 0.7460860196351268
EfficientNetV2-L cross-script AUC difference: -0.09946757111935678


In [18]:
efficientnet_development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=EFFICIENTNET_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

efficientnet_development_embeddings, efficientnet_development_filenames = (
    extract_efficientnet_embeddings(
        efficientnet_model,
        efficientnet_development_loader,
    )
)

development_split_df = split_df[
    split_df[
        "experiment_split"
    ] == "development_train"
]

development_writer_map = dict(
    zip(
        development_split_df[
            "filename"
        ],
        development_split_df[
            "writer"
        ],
    )
)

efficientnet_development_writers = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in efficientnet_development_filenames
    ]
)

print(
    "Development embeddings:",
    efficientnet_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        efficientnet_development_filenames
    ),
)

print(
    "Development writers:",
    len(
        np.unique(
            efficientnet_development_writers
        )
    ),
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            efficientnet_development_writers,
            return_counts=True,
        )[1]
    ),
)

Development embeddings: (904, 1280)
Development filenames: 904
Development writers: 226
Samples per writer: [4]


In [19]:
efficientnet_writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

efficientnet_writer_lda.fit(
    efficientnet_development_embeddings,
    efficientnet_development_writers,
)

efficientnet_lda_development_embeddings = (
    efficientnet_writer_lda.transform(
        efficientnet_development_embeddings
    )
)

efficientnet_lda_validation_embeddings = (
    efficientnet_writer_lda.transform(
        efficientnet_validation_embeddings
    )
)

efficientnet_lda_development_embeddings = (
    efficientnet_lda_development_embeddings
    / np.linalg.norm(
        efficientnet_lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

efficientnet_lda_validation_embeddings = (
    efficientnet_lda_validation_embeddings
    / np.linalg.norm(
        efficientnet_lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

efficientnet_lda_embedding_map = dict(
    zip(
        efficientnet_validation_filenames,
        efficientnet_lda_validation_embeddings,
    )
)

efficientnet_lda_embeddings_a = np.stack(
    [
        efficientnet_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

efficientnet_lda_embeddings_b = np.stack(
    [
        efficientnet_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

efficientnet_lda_scores = np.sum(
    efficientnet_lda_embeddings_a
    * efficientnet_lda_embeddings_b,
    axis=1,
)

efficientnet_lda_auc = roc_auc_score(
    efficientnet_pair_labels,
    efficientnet_lda_scores,
)

efficientnet_lda_eer, efficientnet_lda_threshold = (
    calculate_interpolated_eer(
        efficientnet_pair_labels,
        efficientnet_lda_scores,
    )
)

efficientnet_lda_pair_df = (
    efficientnet_pair_df.copy()
)

efficientnet_lda_pair_df[
    "score"
] = efficientnet_lda_scores

efficientnet_lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = efficientnet_lda_pair_df[
        efficientnet_lda_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    efficientnet_lda_condition_rows.append(
        {
            "condition": condition,
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )

efficientnet_lda_condition_df = pd.DataFrame(
    efficientnet_lda_condition_rows
)

efficientnet_lda_within_macro_auc = (
    efficientnet_lda_condition_df[
        efficientnet_lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

efficientnet_lda_cross_macro_auc = (
    efficientnet_lda_condition_df[
        efficientnet_lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    efficientnet_lda_validation_embeddings.shape[1],
)

print()

print(
    "Frozen EfficientNetV2-L overall AUC:",
    efficientnet_overall_auc,
)

print(
    "EfficientNetV2-L + writer-LDA overall AUC:",
    efficientnet_lda_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "EfficientNetV2-L + writer-LDA EER (%):",
    100.0
    * efficientnet_lda_eer,
)

print()

print(
    efficientnet_lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen EfficientNetV2-L cross-script macro AUC:",
    efficientnet_cross_macro_auc,
)

print(
    "EfficientNetV2-L + writer-LDA cross-script macro AUC:",
    efficientnet_lda_cross_macro_auc,
)

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print()

print(
    "LDA vs batch-alt overall difference:",
    efficientnet_lda_auc
    - current_batch_mean_auc,
)

print(
    "LDA vs batch-alt cross-script difference:",
    efficientnet_lda_cross_macro_auc
    - current_batch_cross_auc,
)

LDA embedding dimension: 225

Frozen EfficientNetV2-L overall AUC: 0.6733993313234385
EfficientNetV2-L + writer-LDA overall AUC: 0.7669261170377243
Current batch-alt mean AUC: 0.7773429715952037

EfficientNetV2-L + writer-LDA EER (%): 30.952380952380953

              condition      auc      eer
   arabic_variable_same 0.837952 0.250000
  english_variable_same 0.864257 0.178571
cross_variable_variable 0.712848 0.348052
    cross_variable_same 0.679974 0.392857
    cross_same_variable 0.752232 0.303571
        cross_same_same 0.758001 0.303571

Frozen EfficientNetV2-L cross-script macro AUC: 0.64661844851577
EfficientNetV2-L + writer-LDA cross-script macro AUC: 0.7257638566790352
Current batch-alt mean cross-script AUC: 0.7460860196351268

LDA vs batch-alt overall difference: -0.010416854557479427
LDA vs batch-alt cross-script difference: -0.020322162956091572


In [21]:
if "efficientnet_model" in globals():
    del efficientnet_model

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

DINOV2_MODEL_NAME = "dinov2_vitl14_reg_frozen"
DINOV2_BATCH_SIZE = 2
DINOV2_INPUT_SIZE = 518

dinov2_model = torch.hub.load(
    "facebookresearch/dinov2:main",
    "dinov2_vitl14_reg",
    trust_repo=True,
    skip_validation=True,
)

dinov2_model = dinov2_model.to(TRAIN_DEVICE)
dinov2_model.eval()

for parameter in dinov2_model.parameters():
    parameter.requires_grad = False

dinov2_mean = torch.tensor(
    [0.485, 0.456, 0.406],
    device=TRAIN_DEVICE,
).view(1, 3, 1, 1)

dinov2_std = torch.tensor(
    [0.229, 0.224, 0.225],
    device=TRAIN_DEVICE,
).view(1, 3, 1, 1)

dummy_image = torch.zeros(
    1,
    3,
    384,
    384,
    device=TRAIN_DEVICE,
)

with torch.no_grad():
    dummy_processed = torch.nn.functional.interpolate(
        dummy_image,
        size=(DINOV2_INPUT_SIZE, DINOV2_INPUT_SIZE),
        mode="bicubic",
        align_corners=False,
    )

    dummy_processed = (
        dummy_processed
        - dinov2_mean
    ) / dinov2_std

    dummy_embedding = dinov2_model(
        dummy_processed
    )

print(
    "Baseline:",
    DINOV2_MODEL_NAME,
)

print(
    "Processed input shape:",
    dummy_processed.shape,
)

print(
    "Embedding shape:",
    dummy_embedding.shape,
)

print(
    "Parameters:",
    sum(
        parameter.numel()
        for parameter in dinov2_model.parameters()
    ),
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in dinov2_model.parameters()
        if parameter.requires_grad
    ),
)

print(
    "GPU memory allocated (GB):",
    torch.cuda.memory_allocated()
    / 1024**3,
)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to C:\Users\com/.cache\torch\hub\main.zip


C:\Users\com/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\com/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\com/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth" to C:\Users\com/.cache\torch\hub\checkpoints\dinov2_vitl14_reg4_pretrain.pth


100.0%


Baseline: dinov2_vitl14_reg_frozen
Processed input shape: torch.Size([1, 3, 518, 518])
Embedding shape: torch.Size([1, 1024])
Parameters: 304372736
Trainable parameters: 0
GPU memory allocated (GB): 1.4748053550720215


In [22]:
dinov2_validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=DINOV2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_dinov2_embeddings(
    model,
    loader,
):
    embeddings = []
    filenames = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            processed_images = torch.nn.functional.interpolate(
                images,
                size=(
                    DINOV2_INPUT_SIZE,
                    DINOV2_INPUT_SIZE,
                ),
                mode="bicubic",
                align_corners=False,
            )

            processed_images = (
                processed_images
                - dinov2_mean
            ) / dinov2_std

            batch_embeddings = model(
                processed_images
            )

            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

            embeddings.append(
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        filenames,
    )


dinov2_validation_embeddings, dinov2_validation_filenames = (
    extract_dinov2_embeddings(
        dinov2_model,
        dinov2_validation_loader,
    )
)

print(
    "Validation embeddings:",
    dinov2_validation_embeddings.shape,
)

print(
    "Validation filenames:",
    len(
        dinov2_validation_filenames
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        dinov2_validation_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Validation embeddings: (224, 1024)
Validation filenames: 224
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 1.6008877754211426


In [23]:
dinov2_embedding_map = dict(
    zip(
        dinov2_validation_filenames,
        dinov2_validation_embeddings,
    )
)

dinov2_embeddings_a = np.stack(
    [
        dinov2_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

dinov2_embeddings_b = np.stack(
    [
        dinov2_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

dinov2_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

dinov2_scores = np.sum(
    dinov2_embeddings_a
    * dinov2_embeddings_b,
    axis=1,
)

dinov2_overall_auc = roc_auc_score(
    dinov2_pair_labels,
    dinov2_scores,
)

dinov2_overall_eer, dinov2_eer_threshold = (
    calculate_interpolated_eer(
        dinov2_pair_labels,
        dinov2_scores,
    )
)

dinov2_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

dinov2_pair_df[
    "score"
] = dinov2_scores

dinov2_pair_df[
    "page_a"
] = dinov2_pair_df[
    "filename_a"
].map(
    filename_page_id
)

dinov2_pair_df[
    "page_b"
] = dinov2_pair_df[
    "filename_b"
].map(
    filename_page_id
)

dinov2_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(page_a),
            int(page_b),
        )
    ]
    for page_a, page_b in zip(
        dinov2_pair_df[
            "page_a"
        ],
        dinov2_pair_df[
            "page_b"
        ],
    )
]

dinov2_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = dinov2_pair_df[
        dinov2_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    dinov2_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

dinov2_condition_df = pd.DataFrame(
    dinov2_condition_rows
)

dinov2_within_macro_auc = (
    dinov2_condition_df[
        dinov2_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

dinov2_cross_macro_auc = (
    dinov2_condition_df[
        dinov2_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "Frozen DINOv2 ViT-L/14-Reg overall AUC:",
    dinov2_overall_auc,
)

print(
    "Frozen DINOv2 ViT-L/14-Reg EER (%):",
    100.0 * dinov2_overall_eer,
)

print(
    "Frozen DINOv2 EER threshold:",
    dinov2_eer_threshold,
)

print()

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print(
    "DINOv2 AUC difference:",
    dinov2_overall_auc
    - current_batch_mean_auc,
)

print()

print(
    dinov2_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen DINOv2 within-script macro AUC:",
    dinov2_within_macro_auc,
)

print(
    "Frozen DINOv2 cross-script macro AUC:",
    dinov2_cross_macro_auc,
)

print()

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print(
    "DINOv2 cross-script AUC difference:",
    dinov2_cross_macro_auc
    - current_batch_cross_auc,
)

Frozen DINOv2 ViT-L/14-Reg overall AUC: 0.5798756055452483
Frozen DINOv2 ViT-L/14-Reg EER (%): 45.53571428571429
Frozen DINOv2 EER threshold: 0.7411571931838989

Current batch-alt mean AUC: 0.7773429715952037
DINOv2 AUC difference: -0.1974673660499554

              condition      auc      eer
   arabic_variable_same 0.760917 0.321429
  english_variable_same 0.717190 0.357143
cross_variable_variable 0.591495 0.466558
    cross_variable_same 0.556879 0.461039
    cross_same_variable 0.596597 0.437662
        cross_same_same 0.599890 0.410714

Frozen DINOv2 within-script macro AUC: 0.7390538033395178
Frozen DINOv2 cross-script macro AUC: 0.5862150684137292

Current batch-alt mean cross-script AUC: 0.7460860196351268
DINOv2 cross-script AUC difference: -0.1598709512213976


In [24]:
dinov2_development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=DINOV2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

dinov2_development_embeddings, dinov2_development_filenames = (
    extract_dinov2_embeddings(
        dinov2_model,
        dinov2_development_loader,
    )
)

development_split_df = split_df[
    split_df[
        "experiment_split"
    ] == "development_train"
]

development_writer_map = dict(
    zip(
        development_split_df[
            "filename"
        ],
        development_split_df[
            "writer"
        ],
    )
)

dinov2_development_writers = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in dinov2_development_filenames
    ]
)

print(
    "Development embeddings:",
    dinov2_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        dinov2_development_filenames
    ),
)

print(
    "Development writers:",
    len(
        np.unique(
            dinov2_development_writers
        )
    ),
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            dinov2_development_writers,
            return_counts=True,
        )[1]
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        dinov2_development_embeddings,
        axis=1,
    ).mean(),
)

Development embeddings: (904, 1024)
Development filenames: 904
Development writers: 226
Samples per writer: [4]
Mean embedding norm: 1.0


In [25]:
dinov2_writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

dinov2_writer_lda.fit(
    dinov2_development_embeddings,
    dinov2_development_writers,
)

dinov2_lda_development_embeddings = (
    dinov2_writer_lda.transform(
        dinov2_development_embeddings
    )
)

dinov2_lda_validation_embeddings = (
    dinov2_writer_lda.transform(
        dinov2_validation_embeddings
    )
)

dinov2_lda_development_embeddings = (
    dinov2_lda_development_embeddings
    / np.linalg.norm(
        dinov2_lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

dinov2_lda_validation_embeddings = (
    dinov2_lda_validation_embeddings
    / np.linalg.norm(
        dinov2_lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

dinov2_lda_embedding_map = dict(
    zip(
        dinov2_validation_filenames,
        dinov2_lda_validation_embeddings,
    )
)

dinov2_lda_embeddings_a = np.stack(
    [
        dinov2_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

dinov2_lda_embeddings_b = np.stack(
    [
        dinov2_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

dinov2_lda_scores = np.sum(
    dinov2_lda_embeddings_a
    * dinov2_lda_embeddings_b,
    axis=1,
)

dinov2_lda_auc = roc_auc_score(
    dinov2_pair_labels,
    dinov2_lda_scores,
)

dinov2_lda_eer, dinov2_lda_threshold = (
    calculate_interpolated_eer(
        dinov2_pair_labels,
        dinov2_lda_scores,
    )
)

dinov2_lda_pair_df = (
    dinov2_pair_df.copy()
)

dinov2_lda_pair_df[
    "score"
] = dinov2_lda_scores

dinov2_lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = dinov2_lda_pair_df[
        dinov2_lda_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    dinov2_lda_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

dinov2_lda_condition_df = pd.DataFrame(
    dinov2_lda_condition_rows
)

dinov2_lda_within_macro_auc = (
    dinov2_lda_condition_df[
        dinov2_lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

dinov2_lda_cross_macro_auc = (
    dinov2_lda_condition_df[
        dinov2_lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    dinov2_lda_validation_embeddings.shape[1],
)

print()

print(
    "Frozen DINOv2 overall AUC:",
    dinov2_overall_auc,
)

print(
    "DINOv2 + writer-LDA overall AUC:",
    dinov2_lda_auc,
)

print(
    "EfficientNetV2-L + writer-LDA overall AUC:",
    efficientnet_lda_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "DINOv2 + writer-LDA EER (%):",
    100.0 * dinov2_lda_eer,
)

print()

print(
    dinov2_lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen DINOv2 cross-script macro AUC:",
    dinov2_cross_macro_auc,
)

print(
    "DINOv2 + writer-LDA cross-script macro AUC:",
    dinov2_lda_cross_macro_auc,
)

print(
    "EfficientNetV2-L + writer-LDA cross-script macro AUC:",
    efficientnet_lda_cross_macro_auc,
)

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print()

print(
    "DINOv2 LDA vs batch-alt overall difference:",
    dinov2_lda_auc
    - current_batch_mean_auc,
)

print(
    "DINOv2 LDA vs batch-alt cross-script difference:",
    dinov2_lda_cross_macro_auc
    - current_batch_cross_auc,
)

LDA embedding dimension: 225

Frozen DINOv2 overall AUC: 0.5798756055452483
DINOv2 + writer-LDA overall AUC: 0.8274522972067615
EfficientNetV2-L + writer-LDA overall AUC: 0.7669261170377243
Current batch-alt mean AUC: 0.7773429715952037

DINOv2 + writer-LDA EER (%): 25.297619047619047

              condition      auc      eer
   arabic_variable_same 0.896614 0.168831
  english_variable_same 0.901281 0.178571
cross_variable_variable 0.811538 0.250000
    cross_variable_same 0.723527 0.339286
    cross_same_variable 0.807218 0.286688
        cross_same_same 0.823748 0.285714

Frozen DINOv2 cross-script macro AUC: 0.5862150684137292
DINOv2 + writer-LDA cross-script macro AUC: 0.7915077110389611
EfficientNetV2-L + writer-LDA cross-script macro AUC: 0.7257638566790352
Current batch-alt mean cross-script AUC: 0.7460860196351268

DINOv2 LDA vs batch-alt overall difference: 0.05010932561155779
DINOv2 LDA vs batch-alt cross-script difference: 0.04542169140383434


In [26]:
BENCHMARK_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modern_baseline_benchmarking"
)

BENCHMARK_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dinov2_embedding_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitl14_reg_embeddings.npz"
)

np.savez_compressed(
    dinov2_embedding_path,
    development_embeddings=dinov2_development_embeddings,
    development_filenames=np.asarray(
        dinov2_development_filenames
    ),
    development_writers=dinov2_development_writers,
    validation_embeddings=dinov2_validation_embeddings,
    validation_filenames=np.asarray(
        dinov2_validation_filenames
    ),
    lda_development_embeddings=dinov2_lda_development_embeddings,
    lda_validation_embeddings=dinov2_lda_validation_embeddings,
)

dinov2_summary_df = pd.DataFrame(
    [
        {
            "model": "dinov2_vitl14_reg_frozen",
            "overall_auc": dinov2_overall_auc,
            "eer": dinov2_overall_eer,
            "within_script_macro_auc": dinov2_within_macro_auc,
            "cross_script_macro_auc": dinov2_cross_macro_auc,
        },
        {
            "model": "dinov2_vitl14_reg_writer_lda",
            "overall_auc": dinov2_lda_auc,
            "eer": dinov2_lda_eer,
            "within_script_macro_auc": dinov2_lda_within_macro_auc,
            "cross_script_macro_auc": dinov2_lda_cross_macro_auc,
        },
    ]
)

dinov2_summary_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitl14_reg_summary.csv"
)

dinov2_condition_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitl14_reg_condition_results.csv"
)

dinov2_summary_df.to_csv(
    dinov2_summary_path,
    index=False,
)

dinov2_condition_save_df = pd.concat(
    [
        dinov2_condition_df.assign(
            model="dinov2_vitl14_reg_frozen"
        ),
        dinov2_lda_condition_df.assign(
            model="dinov2_vitl14_reg_writer_lda"
        ),
    ],
    ignore_index=True,
)

dinov2_condition_save_df.to_csv(
    dinov2_condition_path,
    index=False,
)

print(
    "Embedding file:",
    dinov2_embedding_path,
)

print(
    "Embedding file size (MB):",
    dinov2_embedding_path.stat().st_size
    / 1024**2,
)

print(
    "Summary file:",
    dinov2_summary_path,
)

print(
    "Condition file:",
    dinov2_condition_path,
)

print()

print(
    dinov2_summary_df.round(6).to_string(
        index=False
    )
)

Embedding file: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\dinov2_vitl14_reg_embeddings.npz
Embedding file size (MB): 5.0020599365234375
Summary file: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\dinov2_vitl14_reg_summary.csv
Condition file: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\dinov2_vitl14_reg_condition_results.csv

                       model  overall_auc      eer  within_script_macro_auc  cross_script_macro_auc
    dinov2_vitl14_reg_frozen     0.579876 0.455357                 0.739054                0.586215
dinov2_vitl14_reg_writer_lda     0.827452 0.252976                 0.898948                0.791508


In [27]:
if "dinov2_model" in globals():
    del dinov2_model

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

VIT_H_WEIGHTS = (
    torchvision.models
    .ViT_H_14_Weights
    .IMAGENET1K_SWAG_E2E_V1
)

VIT_H_MODEL_NAME = "vit_h_14_swag_e2e_frozen"
VIT_H_BATCH_SIZE = 1

vit_h_preprocess = VIT_H_WEIGHTS.transforms()

vit_h_model = torchvision.models.vit_h_14(
    weights=VIT_H_WEIGHTS
)

vit_h_model.heads = torch.nn.Identity()

vit_h_model = vit_h_model.to(
    TRAIN_DEVICE
)

vit_h_model.eval()

for parameter in vit_h_model.parameters():
    parameter.requires_grad = False

dummy_image = torch.zeros(
    1,
    3,
    384,
    384,
    device=TRAIN_DEVICE,
)

with torch.no_grad():
    dummy_processed = vit_h_preprocess(
        dummy_image
    )

    dummy_embedding = vit_h_model(
        dummy_processed
    )

print(
    "Baseline:",
    VIT_H_MODEL_NAME,
)

print(
    "Processed input shape:",
    dummy_processed.shape,
)

print(
    "Embedding shape:",
    dummy_embedding.shape,
)

print(
    "Parameters:",
    sum(
        parameter.numel()
        for parameter in vit_h_model.parameters()
    ),
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in vit_h_model.parameters()
        if parameter.requires_grad
    ),
)

print(
    "ImageNet top-1:",
    VIT_H_WEIGHTS.meta[
        "_metrics"
    ][
        "ImageNet-1K"
    ][
        "acc@1"
    ],
)

print(
    "GPU memory allocated (GB):",
    torch.cuda.memory_allocated()
    / 1024**3,
)

Downloading: "https://download.pytorch.org/models/vit_h_14_swag-80465313.pth" to C:\Users\com/.cache\torch\hub\checkpoints\vit_h_14_swag-80465313.pth


100.0%


Baseline: vit_h_14_swag_e2e_frozen
Processed input shape: torch.Size([1, 3, 518, 518])
Embedding shape: torch.Size([1, 1280])
Parameters: 632189440
Trainable parameters: 0
ImageNet top-1: 88.552
GPU memory allocated (GB): 2.759377956390381


In [28]:
VIT_H_BATCH_CANDIDATES = [
    1,
    2,
    4,
    8,
    16,
]

VIT_H_SAFE_MEMORY_GB = 12.0
vit_h_batch_results = []

for batch_size in VIT_H_BATCH_CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        probe_images = torch.zeros(
            batch_size,
            3,
            384,
            384,
            device=TRAIN_DEVICE,
        )

        with torch.inference_mode():
            probe_processed = vit_h_preprocess(
                probe_images
            )

            probe_embeddings = vit_h_model(
                probe_processed
            )

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        vit_h_batch_results.append(
            {
                "batch_size": batch_size,
                "peak_memory_gb": peak_memory_gb,
                "success": True,
            }
        )

        print(
            "Batch:",
            batch_size,
            "| Peak GB:",
            round(
                peak_memory_gb,
                3,
            ),
            "| Embeddings:",
            tuple(
                probe_embeddings.shape
            ),
        )

        del probe_images
        del probe_processed
        del probe_embeddings

    except RuntimeError as error:
        if "out of memory" not in str(
            error
        ).lower():
            raise

        print(
            "Batch:",
            batch_size,
            "| CUDA OOM",
        )

        torch.cuda.empty_cache()
        break

safe_batch_sizes = [
    result[
        "batch_size"
    ]
    for result in vit_h_batch_results
    if (
        result[
            "success"
        ]
        and result[
            "peak_memory_gb"
        ] <= VIT_H_SAFE_MEMORY_GB
    )
]

VIT_H_BATCH_SIZE = max(
    safe_batch_sizes
)

torch.cuda.empty_cache()

print()
print(
    "Selected ViT-H batch size:",
    VIT_H_BATCH_SIZE,
)

Batch: 1 | Peak GB: 2.849 | Embeddings: (1, 1280)
Batch: 2 | Peak GB: 2.943 | Embeddings: (2, 1280)
Batch: 4 | Peak GB: 3.12 | Embeddings: (4, 1280)
Batch: 8 | Peak GB: 3.479 | Embeddings: (8, 1280)
Batch: 16 | Peak GB: 4.199 | Embeddings: (16, 1280)

Selected ViT-H batch size: 16


In [29]:
for batch_size in [32, 64, 96]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        probe_images = torch.zeros(
            batch_size,
            3,
            384,
            384,
            device=TRAIN_DEVICE,
        )

        with torch.inference_mode():
            probe_processed = vit_h_preprocess(
                probe_images
            )

            probe_embeddings = vit_h_model(
                probe_processed
            )

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        print(
            "Batch:",
            batch_size,
            "| Peak GB:",
            round(
                peak_memory_gb,
                3,
            ),
            "| Embeddings:",
            tuple(
                probe_embeddings.shape
            ),
        )

        del probe_images
        del probe_processed
        del probe_embeddings

    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise

        print(
            "Batch:",
            batch_size,
            "| CUDA OOM",
        )

        torch.cuda.empty_cache()
        break

Batch: 32 | Peak GB: 5.626 | Embeddings: (32, 1280)
Batch: 64 | Peak GB: 8.492 | Embeddings: (64, 1280)
Batch: 96 | Peak GB: 11.358 | Embeddings: (96, 1280)


In [30]:
VIT_H_BATCH_SIZE = 96

print(
    "Final ViT-H batch size:",
    VIT_H_BATCH_SIZE,
)

Final ViT-H batch size: 96


In [31]:
vit_h_validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=VIT_H_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_vit_h_embeddings(
    model,
    loader,
):
    embeddings = []
    filenames = []

    model.eval()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            processed_images = vit_h_preprocess(
                images
            )

            batch_embeddings = model(
                processed_images
            )

            batch_embeddings = (
                torch.nn.functional.normalize(
                    batch_embeddings,
                    p=2,
                    dim=1,
                )
            )

            embeddings.append(
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        filenames,
    )


vit_h_validation_embeddings, vit_h_validation_filenames = (
    extract_vit_h_embeddings(
        vit_h_model,
        vit_h_validation_loader,
    )
)

print(
    "Selected batch size:",
    VIT_H_BATCH_SIZE,
)

print(
    "Validation embeddings:",
    vit_h_validation_embeddings.shape,
)

print(
    "Validation filenames:",
    len(
        vit_h_validation_filenames
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        vit_h_validation_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Selected batch size: 96
Validation embeddings: (224, 1280)
Validation filenames: 224
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 11.35869026184082


In [32]:
vit_h_embedding_map = dict(
    zip(
        vit_h_validation_filenames,
        vit_h_validation_embeddings,
    )
)

vit_h_embeddings_a = np.stack(
    [
        vit_h_embedding_map[filename]
        for filename in validation_pairs_df["filename_a"]
    ]
)

vit_h_embeddings_b = np.stack(
    [
        vit_h_embedding_map[filename]
        for filename in validation_pairs_df["filename_b"]
    ]
)

vit_h_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

vit_h_scores = np.sum(
    vit_h_embeddings_a
    * vit_h_embeddings_b,
    axis=1,
)

vit_h_overall_auc = roc_auc_score(
    vit_h_pair_labels,
    vit_h_scores,
)

vit_h_overall_eer, vit_h_eer_threshold = (
    calculate_interpolated_eer(
        vit_h_pair_labels,
        vit_h_scores,
    )
)

vit_h_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

vit_h_pair_df[
    "score"
] = vit_h_scores

vit_h_pair_df[
    "page_a"
] = vit_h_pair_df[
    "filename_a"
].map(
    filename_page_id
)

vit_h_pair_df[
    "page_b"
] = vit_h_pair_df[
    "filename_b"
].map(
    filename_page_id
)

vit_h_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(page_a),
            int(page_b),
        )
    ]
    for page_a, page_b in zip(
        vit_h_pair_df["page_a"],
        vit_h_pair_df["page_b"],
    )
]

vit_h_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = vit_h_pair_df[
        vit_h_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    vit_h_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

vit_h_condition_df = pd.DataFrame(
    vit_h_condition_rows
)

vit_h_within_macro_auc = (
    vit_h_condition_df[
        vit_h_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

vit_h_cross_macro_auc = (
    vit_h_condition_df[
        vit_h_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "Frozen ViT-H/14 SWAG overall AUC:",
    vit_h_overall_auc,
)

print(
    "Frozen ViT-H/14 SWAG EER (%):",
    100.0 * vit_h_overall_eer,
)

print(
    "Frozen ViT-H/14 SWAG EER threshold:",
    vit_h_eer_threshold,
)

print()

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print(
    "DINOv2 + writer-LDA overall AUC:",
    dinov2_lda_auc,
)

print(
    "ViT-H AUC difference vs batch-alt:",
    vit_h_overall_auc
    - current_batch_mean_auc,
)

print()

print(
    vit_h_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT-H within-script macro AUC:",
    vit_h_within_macro_auc,
)

print(
    "Frozen ViT-H cross-script macro AUC:",
    vit_h_cross_macro_auc,
)

print()

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print(
    "DINOv2 + writer-LDA cross-script macro AUC:",
    dinov2_lda_cross_macro_auc,
)

Frozen ViT-H/14 SWAG overall AUC: 0.5731711245104101
Frozen ViT-H/14 SWAG EER (%): 45.52489177489177
Frozen ViT-H/14 SWAG EER threshold: 0.6070864720778032

Current batch-alt mean AUC: 0.7773429715952037
DINOv2 + writer-LDA overall AUC: 0.8274522972067615
ViT-H AUC difference vs batch-alt: -0.20417184708479363

              condition      auc      eer
   arabic_variable_same 0.738016 0.321429
  english_variable_same 0.709085 0.339286
cross_variable_variable 0.586445 0.414286
    cross_variable_same 0.555804 0.458766
    cross_same_variable 0.594799 0.446429
        cross_same_same 0.628542 0.410714

Frozen ViT-H within-script macro AUC: 0.7235505565862708
Frozen ViT-H cross-script macro AUC: 0.5913975533395176

Current batch-alt mean cross-script AUC: 0.7460860196351268
DINOv2 + writer-LDA cross-script macro AUC: 0.7915077110389611


In [33]:
vit_h_development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=VIT_H_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

vit_h_development_embeddings, vit_h_development_filenames = (
    extract_vit_h_embeddings(
        vit_h_model,
        vit_h_development_loader,
    )
)

development_split_df = split_df[
    split_df[
        "experiment_split"
    ] == "development_train"
]

development_writer_map = dict(
    zip(
        development_split_df[
            "filename"
        ],
        development_split_df[
            "writer"
        ],
    )
)

vit_h_development_writers = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in vit_h_development_filenames
    ]
)

print(
    "Development embeddings:",
    vit_h_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        vit_h_development_filenames
    ),
)

print(
    "Development writers:",
    len(
        np.unique(
            vit_h_development_writers
        )
    ),
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            vit_h_development_writers,
            return_counts=True,
        )[1]
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        vit_h_development_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Development embeddings: (904, 1280)
Development filenames: 904
Development writers: 226
Samples per writer: [4]
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 11.35869026184082


In [34]:
vit_h_writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

vit_h_writer_lda.fit(
    vit_h_development_embeddings,
    vit_h_development_writers,
)

vit_h_lda_development_embeddings = (
    vit_h_writer_lda.transform(
        vit_h_development_embeddings
    )
)

vit_h_lda_validation_embeddings = (
    vit_h_writer_lda.transform(
        vit_h_validation_embeddings
    )
)

vit_h_lda_development_embeddings = (
    vit_h_lda_development_embeddings
    / np.linalg.norm(
        vit_h_lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

vit_h_lda_validation_embeddings = (
    vit_h_lda_validation_embeddings
    / np.linalg.norm(
        vit_h_lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

vit_h_lda_embedding_map = dict(
    zip(
        vit_h_validation_filenames,
        vit_h_lda_validation_embeddings,
    )
)

vit_h_lda_embeddings_a = np.stack(
    [
        vit_h_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

vit_h_lda_embeddings_b = np.stack(
    [
        vit_h_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

vit_h_lda_scores = np.sum(
    vit_h_lda_embeddings_a
    * vit_h_lda_embeddings_b,
    axis=1,
)

vit_h_lda_auc = roc_auc_score(
    vit_h_pair_labels,
    vit_h_lda_scores,
)

vit_h_lda_eer, vit_h_lda_threshold = (
    calculate_interpolated_eer(
        vit_h_pair_labels,
        vit_h_lda_scores,
    )
)

vit_h_lda_pair_df = (
    vit_h_pair_df.copy()
)

vit_h_lda_pair_df[
    "score"
] = vit_h_lda_scores

vit_h_lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = vit_h_lda_pair_df[
        vit_h_lda_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    vit_h_lda_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

vit_h_lda_condition_df = pd.DataFrame(
    vit_h_lda_condition_rows
)

vit_h_lda_within_macro_auc = (
    vit_h_lda_condition_df[
        vit_h_lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

vit_h_lda_cross_macro_auc = (
    vit_h_lda_condition_df[
        vit_h_lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    vit_h_lda_validation_embeddings.shape[1],
)

print()

print(
    "Frozen ViT-H overall AUC:",
    vit_h_overall_auc,
)

print(
    "ViT-H + writer-LDA overall AUC:",
    vit_h_lda_auc,
)

print(
    "DINOv2 + writer-LDA overall AUC:",
    dinov2_lda_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "ViT-H + writer-LDA EER (%):",
    100.0 * vit_h_lda_eer,
)

print()

print(
    vit_h_lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT-H cross-script macro AUC:",
    vit_h_cross_macro_auc,
)

print(
    "ViT-H + writer-LDA cross-script macro AUC:",
    vit_h_lda_cross_macro_auc,
)

print(
    "DINOv2 + writer-LDA cross-script macro AUC:",
    dinov2_lda_cross_macro_auc,
)

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print()

print(
    "ViT-H LDA vs DINOv2 LDA overall difference:",
    vit_h_lda_auc
    - dinov2_lda_auc,
)

print(
    "ViT-H LDA vs DINOv2 LDA cross-script difference:",
    vit_h_lda_cross_macro_auc
    - dinov2_lda_cross_macro_auc,
)

LDA embedding dimension: 225

Frozen ViT-H overall AUC: 0.5731711245104101
ViT-H + writer-LDA overall AUC: 0.6879805710162853
DINOv2 + writer-LDA overall AUC: 0.8274522972067615
Current batch-alt mean AUC: 0.7773429715952037

ViT-H + writer-LDA EER (%): 36.904761904761905

              condition      auc      eer
   arabic_variable_same 0.916338 0.164286
  english_variable_same 0.686468 0.386688
cross_variable_variable 0.619231 0.371429
    cross_variable_same 0.638596 0.415584
    cross_same_variable 0.601560 0.428571
        cross_same_same 0.658244 0.392857

Frozen ViT-H cross-script macro AUC: 0.5913975533395176
ViT-H + writer-LDA cross-script macro AUC: 0.6294077574211503
DINOv2 + writer-LDA cross-script macro AUC: 0.7915077110389611
Current batch-alt mean cross-script AUC: 0.7460860196351268

ViT-H LDA vs DINOv2 LDA overall difference: -0.1394717261904762
ViT-H LDA vs DINOv2 LDA cross-script difference: -0.16209995361781082


In [35]:
vit_h_embedding_path = (
    BENCHMARK_REPORT_DIR
    / "vit_h_14_swag_e2e_embeddings.npz"
)

np.savez_compressed(
    vit_h_embedding_path,
    development_embeddings=vit_h_development_embeddings,
    development_filenames=np.asarray(
        vit_h_development_filenames
    ),
    development_writers=vit_h_development_writers,
    validation_embeddings=vit_h_validation_embeddings,
    validation_filenames=np.asarray(
        vit_h_validation_filenames
    ),
    lda_development_embeddings=vit_h_lda_development_embeddings,
    lda_validation_embeddings=vit_h_lda_validation_embeddings,
)

vit_h_summary_df = pd.DataFrame(
    [
        {
            "model": "vit_h_14_swag_e2e_frozen",
            "overall_auc": vit_h_overall_auc,
            "eer": vit_h_overall_eer,
            "within_script_macro_auc": vit_h_within_macro_auc,
            "cross_script_macro_auc": vit_h_cross_macro_auc,
        },
        {
            "model": "vit_h_14_swag_e2e_writer_lda",
            "overall_auc": vit_h_lda_auc,
            "eer": vit_h_lda_eer,
            "within_script_macro_auc": vit_h_lda_within_macro_auc,
            "cross_script_macro_auc": vit_h_lda_cross_macro_auc,
        },
    ]
)

vit_h_summary_path = (
    BENCHMARK_REPORT_DIR
    / "vit_h_14_swag_e2e_summary.csv"
)

vit_h_condition_path = (
    BENCHMARK_REPORT_DIR
    / "vit_h_14_swag_e2e_condition_results.csv"
)

vit_h_summary_df.to_csv(
    vit_h_summary_path,
    index=False,
)

vit_h_condition_save_df = pd.concat(
    [
        vit_h_condition_df.assign(
            model="vit_h_14_swag_e2e_frozen"
        ),
        vit_h_lda_condition_df.assign(
            model="vit_h_14_swag_e2e_writer_lda"
        ),
    ],
    ignore_index=True,
)

vit_h_condition_save_df.to_csv(
    vit_h_condition_path,
    index=False,
)

print(
    "Embedding file:",
    vit_h_embedding_path,
)

print(
    "Embedding file size (MB):",
    vit_h_embedding_path.stat().st_size
    / 1024**2,
)

print(
    vit_h_summary_df
    .round(6)
    .to_string(
        index=False
    )
)

Embedding file: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\vit_h_14_swag_e2e_embeddings.npz
Embedding file size (MB): 6.0240373611450195
                       model  overall_auc      eer  within_script_macro_auc  cross_script_macro_auc
    vit_h_14_swag_e2e_frozen     0.573171 0.455249                 0.723551                0.591398
vit_h_14_swag_e2e_writer_lda     0.687981 0.369048                 0.801403                0.629408


In [39]:
benchmark_summary_df = pd.DataFrame(
    [
        {
            "model": "batch_alt_adversarial_mean",
            "adaptation": "task_specific_training",
            "overall_auc": current_batch_mean_auc,
            "eer": np.nan,
            "cross_script_macro_auc": current_batch_cross_auc,
        },
        {
            "model": "vit_b_16_swag_e2e",
            "adaptation": "frozen",
            "overall_auc": vit_overall_auc,
            "eer": vit_overall_eer,
            "cross_script_macro_auc": vit_cross_macro_auc,
        },
        {
            "model": "vit_b_16_swag_e2e",
            "adaptation": "writer_lda",
            "overall_auc": lda_overall_auc,
            "eer": lda_overall_eer,
            "cross_script_macro_auc": lda_cross_macro_auc,
        },
        {
            "model": "efficientnet_v2_l",
            "adaptation": "frozen",
            "overall_auc": efficientnet_overall_auc,
            "eer": efficientnet_overall_eer,
            "cross_script_macro_auc": efficientnet_cross_macro_auc,
        },
        {
            "model": "efficientnet_v2_l",
            "adaptation": "writer_lda",
            "overall_auc": efficientnet_lda_auc,
            "eer": efficientnet_lda_eer,
            "cross_script_macro_auc": efficientnet_lda_cross_macro_auc,
        },
        {
            "model": "dinov2_vitl14_reg",
            "adaptation": "frozen",
            "overall_auc": dinov2_overall_auc,
            "eer": dinov2_overall_eer,
            "cross_script_macro_auc": dinov2_cross_macro_auc,
        },
        {
            "model": "dinov2_vitl14_reg",
            "adaptation": "writer_lda",
            "overall_auc": dinov2_lda_auc,
            "eer": dinov2_lda_eer,
            "cross_script_macro_auc": dinov2_lda_cross_macro_auc,
        },
        {
            "model": "vit_h_14_swag_e2e",
            "adaptation": "frozen",
            "overall_auc": vit_h_overall_auc,
            "eer": vit_h_overall_eer,
            "cross_script_macro_auc": vit_h_cross_macro_auc,
        },
        {
            "model": "vit_h_14_swag_e2e",
            "adaptation": "writer_lda",
            "overall_auc": vit_h_lda_auc,
            "eer": vit_h_lda_eer,
            "cross_script_macro_auc": vit_h_lda_cross_macro_auc,
        },
    ]
)

benchmark_summary_df = (
    benchmark_summary_df
    .sort_values(
        "overall_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

benchmark_summary_path = (
    BENCHMARK_REPORT_DIR
    / "modern_baseline_summary.csv"
)

benchmark_summary_df.to_csv(
    benchmark_summary_path,
    index=False,
)

print(
    benchmark_summary_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Saved:",
    benchmark_summary_path,
)

                     model             adaptation  overall_auc      eer  cross_script_macro_auc
         dinov2_vitl14_reg             writer_lda     0.827452 0.252976                0.791508
batch_alt_adversarial_mean task_specific_training     0.777343      NaN                0.746086
         efficientnet_v2_l             writer_lda     0.766926 0.309524                0.725764
         vit_b_16_swag_e2e             writer_lda     0.703563 0.350000                0.655685
         vit_h_14_swag_e2e             writer_lda     0.687981 0.369048                0.629408
         efficientnet_v2_l                 frozen     0.673399 0.375000                0.646618
         vit_b_16_swag_e2e                 frozen     0.671342 0.383929                0.647814
         dinov2_vitl14_reg                 frozen     0.579876 0.455357                0.586215
         vit_h_14_swag_e2e                 frozen     0.573171 0.455249                0.591398

Saved: C:\Users\com\Documents\handwriti

In [40]:
if "vit_h_model" in globals():
    del vit_h_model

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

DINOV2_G_MODEL_NAME = "dinov2_vitg14_reg_frozen"
DINOV2_G_INPUT_SIZE = 518

dinov2_g_model = torch.hub.load(
    "facebookresearch/dinov2:main",
    "dinov2_vitg14_reg",
    trust_repo=True,
    skip_validation=True,
)

dinov2_g_model = dinov2_g_model.to(
    TRAIN_DEVICE
)

dinov2_g_model.eval()

for parameter in dinov2_g_model.parameters():
    parameter.requires_grad = False

dummy_image = torch.zeros(
    1,
    3,
    384,
    384,
    device=TRAIN_DEVICE,
)

with torch.inference_mode():
    dummy_processed = torch.nn.functional.interpolate(
        dummy_image,
        size=(
            DINOV2_G_INPUT_SIZE,
            DINOV2_G_INPUT_SIZE,
        ),
        mode="bicubic",
        align_corners=False,
    )

    dummy_processed = (
        dummy_processed
        - dinov2_mean
    ) / dinov2_std

    dummy_embedding = dinov2_g_model(
        dummy_processed
    )

print(
    "Baseline:",
    DINOV2_G_MODEL_NAME,
)

print(
    "Processed input shape:",
    dummy_processed.shape,
)

print(
    "Embedding shape:",
    dummy_embedding.shape,
)

print(
    "Parameters:",
    sum(
        parameter.numel()
        for parameter in dinov2_g_model.parameters()
    ),
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in dinov2_g_model.parameters()
        if parameter.requires_grad
    ),
)

print(
    "GPU memory allocated (GB):",
    torch.cuda.memory_allocated()
    / 1024**3,
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Using cache found in C:\Users\com/.cache\torch\hub\facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitg14/dinov2_vitg14_reg4_pretrain.pth" to C:\Users\com/.cache\torch\hub\checkpoints\dinov2_vitg14_reg4_pretrain.pth


100.0%


Baseline: dinov2_vitg14_reg_frozen
Processed input shape: torch.Size([1, 3, 518, 518])
Embedding shape: torch.Size([1, 1536])
Parameters: 1136486912
Trainable parameters: 0
GPU memory allocated (GB): 4.616682529449463
Peak GPU memory allocated (GB): 4.723320484161377


In [41]:
DINOV2_G_BATCH_CANDIDATES = [
    2,
    4,
    8,
]

DINOV2_G_SAFE_MEMORY_GB = 12.0
dinov2_g_batch_results = []

if "dummy_image" in globals():
    del dummy_image

if "dummy_processed" in globals():
    del dummy_processed

if "dummy_embedding" in globals():
    del dummy_embedding

torch.cuda.empty_cache()

for batch_size in DINOV2_G_BATCH_CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        probe_images = torch.zeros(
            batch_size,
            3,
            384,
            384,
            device=TRAIN_DEVICE,
        )

        with torch.inference_mode():
            probe_processed = torch.nn.functional.interpolate(
                probe_images,
                size=(
                    DINOV2_G_INPUT_SIZE,
                    DINOV2_G_INPUT_SIZE,
                ),
                mode="bicubic",
                align_corners=False,
            )

            probe_processed = (
                probe_processed
                - dinov2_mean
            ) / dinov2_std

            probe_embeddings = dinov2_g_model(
                probe_processed
            )

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        dinov2_g_batch_results.append(
            {
                "batch_size": batch_size,
                "peak_memory_gb": peak_memory_gb,
            }
        )

        print(
            "Batch:",
            batch_size,
            "| Peak GB:",
            round(
                peak_memory_gb,
                3,
            ),
            "| Embeddings:",
            tuple(
                probe_embeddings.shape
            ),
        )

        del probe_images
        del probe_processed
        del probe_embeddings

        if peak_memory_gb > DINOV2_G_SAFE_MEMORY_GB:
            break

    except RuntimeError as error:
        if "out of memory" not in str(
            error
        ).lower():
            raise

        print(
            "Batch:",
            batch_size,
            "| CUDA OOM",
        )

        torch.cuda.empty_cache()
        break

safe_batch_sizes = [
    1
]

safe_batch_sizes.extend(
    [
        result["batch_size"]
        for result in dinov2_g_batch_results
        if result["peak_memory_gb"] <= DINOV2_G_SAFE_MEMORY_GB
    ]
)

DINOV2_G_BATCH_SIZE = max(
    safe_batch_sizes
)

torch.cuda.empty_cache()

print()
print(
    "Selected DINOv2-G batch size:",
    DINOV2_G_BATCH_SIZE,
)

Batch: 2 | Peak GB: 4.829 | Embeddings: (2, 1536)
Batch: 4 | Peak GB: 5.052 | Embeddings: (4, 1536)
Batch: 8 | Peak GB: 5.502 | Embeddings: (8, 1536)

Selected DINOv2-G batch size: 8


In [42]:
for batch_size in [16, 32]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        probe_images = torch.zeros(
            batch_size,
            3,
            384,
            384,
            device=TRAIN_DEVICE,
        )

        with torch.inference_mode():
            probe_processed = torch.nn.functional.interpolate(
                probe_images,
                size=(
                    DINOV2_G_INPUT_SIZE,
                    DINOV2_G_INPUT_SIZE,
                ),
                mode="bicubic",
                align_corners=False,
            )

            probe_processed = (
                probe_processed
                - dinov2_mean
            ) / dinov2_std

            probe_embeddings = dinov2_g_model(
                probe_processed
            )

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        print(
            "Batch:",
            batch_size,
            "| Peak GB:",
            round(
                peak_memory_gb,
                3,
            ),
            "| Embeddings:",
            tuple(
                probe_embeddings.shape
            ),
        )

        del probe_images
        del probe_processed
        del probe_embeddings

        if peak_memory_gb > 12.0:
            break

    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise

        print(
            "Batch:",
            batch_size,
            "| CUDA OOM",
        )

        torch.cuda.empty_cache()
        break

Batch: 16 | Peak GB: 6.4 | Embeddings: (16, 1536)
Batch: 32 | Peak GB: 8.193 | Embeddings: (32, 1536)


In [43]:
DINOV2_G_BATCH_SIZE = 32

print(
    "Final DINOv2-G batch size:",
    DINOV2_G_BATCH_SIZE,
)

Final DINOv2-G batch size: 32


In [44]:
dinov2_g_validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=DINOV2_G_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_dinov2_g_embeddings(
    model,
    loader,
):
    embeddings = []
    filenames = []

    model.eval()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            processed_images = torch.nn.functional.interpolate(
                images,
                size=(
                    DINOV2_G_INPUT_SIZE,
                    DINOV2_G_INPUT_SIZE,
                ),
                mode="bicubic",
                align_corners=False,
            )

            processed_images = (
                processed_images
                - dinov2_mean
            ) / dinov2_std

            batch_embeddings = dinov2_g_model(
                processed_images
            )

            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

            embeddings.append(
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        filenames,
    )


dinov2_g_validation_embeddings, dinov2_g_validation_filenames = (
    extract_dinov2_g_embeddings(
        dinov2_g_model,
        dinov2_g_validation_loader,
    )
)

print(
    "Selected batch size:",
    DINOV2_G_BATCH_SIZE,
)

print(
    "Validation embeddings:",
    dinov2_g_validation_embeddings.shape,
)

print(
    "Validation filenames:",
    len(
        dinov2_g_validation_filenames
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        dinov2_g_validation_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Selected batch size: 32
Validation embeddings: (224, 1536)
Validation filenames: 224
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 8.19380235671997


In [45]:
dinov2_g_embedding_map = dict(
    zip(
        dinov2_g_validation_filenames,
        dinov2_g_validation_embeddings,
    )
)

dinov2_g_embeddings_a = np.stack(
    [
        dinov2_g_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

dinov2_g_embeddings_b = np.stack(
    [
        dinov2_g_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

dinov2_g_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

dinov2_g_scores = np.sum(
    dinov2_g_embeddings_a
    * dinov2_g_embeddings_b,
    axis=1,
)

dinov2_g_overall_auc = roc_auc_score(
    dinov2_g_pair_labels,
    dinov2_g_scores,
)

dinov2_g_overall_eer, dinov2_g_eer_threshold = (
    calculate_interpolated_eer(
        dinov2_g_pair_labels,
        dinov2_g_scores,
    )
)

dinov2_g_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

dinov2_g_pair_df[
    "score"
] = dinov2_g_scores

dinov2_g_pair_df[
    "page_a"
] = dinov2_g_pair_df[
    "filename_a"
].map(
    filename_page_id
)

dinov2_g_pair_df[
    "page_b"
] = dinov2_g_pair_df[
    "filename_b"
].map(
    filename_page_id
)

dinov2_g_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(page_a),
            int(page_b),
        )
    ]
    for page_a, page_b in zip(
        dinov2_g_pair_df[
            "page_a"
        ],
        dinov2_g_pair_df[
            "page_b"
        ],
    )
]

dinov2_g_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = dinov2_g_pair_df[
        dinov2_g_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    dinov2_g_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

dinov2_g_condition_df = pd.DataFrame(
    dinov2_g_condition_rows
)

dinov2_g_within_macro_auc = (
    dinov2_g_condition_df[
        dinov2_g_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

dinov2_g_cross_macro_auc = (
    dinov2_g_condition_df[
        dinov2_g_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "Frozen DINOv2-G overall AUC:",
    dinov2_g_overall_auc,
)

print(
    "Frozen DINOv2-G EER (%):",
    100.0
    * dinov2_g_overall_eer,
)

print()

print(
    "DINOv2-L + writer-LDA overall AUC:",
    dinov2_lda_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    dinov2_g_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen DINOv2-G within-script macro AUC:",
    dinov2_g_within_macro_auc,
)

print(
    "Frozen DINOv2-G cross-script macro AUC:",
    dinov2_g_cross_macro_auc,
)

print()

print(
    "DINOv2-L + writer-LDA cross-script macro AUC:",
    dinov2_lda_cross_macro_auc,
)

Frozen DINOv2-G overall AUC: 0.5828865021387344
Frozen DINOv2-G EER (%): 45.048701298701296

DINOv2-L + writer-LDA overall AUC: 0.8274522972067615
Current batch-alt mean AUC: 0.7773429715952037

              condition      auc      eer
   arabic_variable_same 0.759868 0.339286
  english_variable_same 0.703235 0.375000
cross_variable_variable 0.588294 0.464286
    cross_variable_same 0.551606 0.465260
    cross_same_variable 0.615695 0.409740
        cross_same_same 0.603711 0.428571

Frozen DINOv2-G within-script macro AUC: 0.7315514842300557
Frozen DINOv2-G cross-script macro AUC: 0.5898263566790353

DINOv2-L + writer-LDA cross-script macro AUC: 0.7915077110389611


In [46]:
dinov2_g_development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=DINOV2_G_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

dinov2_g_development_embeddings, dinov2_g_development_filenames = (
    extract_dinov2_g_embeddings(
        dinov2_g_model,
        dinov2_g_development_loader,
    )
)

development_split_df = split_df[
    split_df[
        "experiment_split"
    ] == "development_train"
]

development_writer_map = dict(
    zip(
        development_split_df[
            "filename"
        ],
        development_split_df[
            "writer"
        ],
    )
)

dinov2_g_development_writers = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in dinov2_g_development_filenames
    ]
)

print(
    "Development embeddings:",
    dinov2_g_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        dinov2_g_development_filenames
    ),
)

print(
    "Development writers:",
    len(
        np.unique(
            dinov2_g_development_writers
        )
    ),
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            dinov2_g_development_writers,
            return_counts=True,
        )[1]
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        dinov2_g_development_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Peak GPU memory allocated (GB):",
    torch.cuda.max_memory_allocated()
    / 1024**3,
)

Development embeddings: (904, 1536)
Development filenames: 904
Development writers: 226
Samples per writer: [4]
Mean embedding norm: 1.0
Peak GPU memory allocated (GB): 8.19380235671997


In [47]:
dinov2_g_writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

dinov2_g_writer_lda.fit(
    dinov2_g_development_embeddings,
    dinov2_g_development_writers,
)

dinov2_g_lda_development_embeddings = (
    dinov2_g_writer_lda.transform(
        dinov2_g_development_embeddings
    )
)

dinov2_g_lda_validation_embeddings = (
    dinov2_g_writer_lda.transform(
        dinov2_g_validation_embeddings
    )
)

dinov2_g_lda_development_embeddings = (
    dinov2_g_lda_development_embeddings
    / np.linalg.norm(
        dinov2_g_lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

dinov2_g_lda_validation_embeddings = (
    dinov2_g_lda_validation_embeddings
    / np.linalg.norm(
        dinov2_g_lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

dinov2_g_lda_embedding_map = dict(
    zip(
        dinov2_g_validation_filenames,
        dinov2_g_lda_validation_embeddings,
    )
)

dinov2_g_lda_embeddings_a = np.stack(
    [
        dinov2_g_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_a"
        ]
    ]
)

dinov2_g_lda_embeddings_b = np.stack(
    [
        dinov2_g_lda_embedding_map[
            filename
        ]
        for filename in validation_pairs_df[
            "filename_b"
        ]
    ]
)

dinov2_g_lda_scores = np.sum(
    dinov2_g_lda_embeddings_a
    * dinov2_g_lda_embeddings_b,
    axis=1,
)

dinov2_g_lda_auc = roc_auc_score(
    dinov2_g_pair_labels,
    dinov2_g_lda_scores,
)

dinov2_g_lda_eer, dinov2_g_lda_threshold = (
    calculate_interpolated_eer(
        dinov2_g_pair_labels,
        dinov2_g_lda_scores,
    )
)

dinov2_g_lda_pair_df = (
    dinov2_g_pair_df.copy()
)

dinov2_g_lda_pair_df[
    "score"
] = dinov2_g_lda_scores

dinov2_g_lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = dinov2_g_lda_pair_df[
        dinov2_g_lda_pair_df[
            "condition"
        ] == condition
    ]

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    dinov2_g_lda_condition_rows.append(
        {
            "condition": condition,
            "auc": float(auc),
            "eer": float(eer),
        }
    )

dinov2_g_lda_condition_df = pd.DataFrame(
    dinov2_g_lda_condition_rows
)

dinov2_g_lda_within_macro_auc = (
    dinov2_g_lda_condition_df[
        dinov2_g_lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

dinov2_g_lda_cross_macro_auc = (
    dinov2_g_lda_condition_df[
        dinov2_g_lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    dinov2_g_lda_validation_embeddings.shape[1],
)

print()

print(
    "Frozen DINOv2-G overall AUC:",
    dinov2_g_overall_auc,
)

print(
    "DINOv2-G + writer-LDA overall AUC:",
    dinov2_g_lda_auc,
)

print(
    "DINOv2-L + writer-LDA overall AUC:",
    dinov2_lda_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "DINOv2-G + writer-LDA EER (%):",
    100.0 * dinov2_g_lda_eer,
)

print()

print(
    dinov2_g_lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "DINOv2-G + writer-LDA within-script macro AUC:",
    dinov2_g_lda_within_macro_auc,
)

print(
    "DINOv2-G + writer-LDA cross-script macro AUC:",
    dinov2_g_lda_cross_macro_auc,
)

print(
    "DINOv2-L + writer-LDA cross-script macro AUC:",
    dinov2_lda_cross_macro_auc,
)

print()

print(
    "Giant vs Large overall difference:",
    dinov2_g_lda_auc
    - dinov2_lda_auc,
)

print(
    "Giant vs Large cross-script difference:",
    dinov2_g_lda_cross_macro_auc
    - dinov2_lda_cross_macro_auc,
)

LDA embedding dimension: 225

Frozen DINOv2-G overall AUC: 0.5828865021387344
DINOv2-G + writer-LDA overall AUC: 0.8656665507111936
DINOv2-L + writer-LDA overall AUC: 0.8274522972067615
Current batch-alt mean AUC: 0.7773429715952037

DINOv2-G + writer-LDA EER (%): 21.42857142857143

              condition      auc      eer
   arabic_variable_same 0.915735 0.178571
  english_variable_same 0.968263 0.107143
cross_variable_variable 0.801513 0.285714
    cross_variable_same 0.777267 0.285714
    cross_same_variable 0.841721 0.231818
        cross_same_same 0.890584 0.178571

DINOv2-G + writer-LDA within-script macro AUC: 0.9419990723562152
DINOv2-G + writer-LDA cross-script macro AUC: 0.82777133580705
DINOv2-L + writer-LDA cross-script macro AUC: 0.7915077110389611

Giant vs Large overall difference: 0.03821425350443208
Giant vs Large cross-script difference: 0.036263624768088865


In [48]:
dinov2_g_embedding_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitg14_reg_embeddings.npz"
)

np.savez_compressed(
    dinov2_g_embedding_path,
    development_embeddings=dinov2_g_development_embeddings,
    development_filenames=np.asarray(
        dinov2_g_development_filenames
    ),
    development_writers=dinov2_g_development_writers,
    validation_embeddings=dinov2_g_validation_embeddings,
    validation_filenames=np.asarray(
        dinov2_g_validation_filenames
    ),
    lda_development_embeddings=dinov2_g_lda_development_embeddings,
    lda_validation_embeddings=dinov2_g_lda_validation_embeddings,
)

dinov2_g_summary_df = pd.DataFrame(
    [
        {
            "model": "dinov2_vitg14_reg_frozen",
            "overall_auc": dinov2_g_overall_auc,
            "eer": dinov2_g_overall_eer,
            "within_script_macro_auc": dinov2_g_within_macro_auc,
            "cross_script_macro_auc": dinov2_g_cross_macro_auc,
        },
        {
            "model": "dinov2_vitg14_reg_writer_lda",
            "overall_auc": dinov2_g_lda_auc,
            "eer": dinov2_g_lda_eer,
            "within_script_macro_auc": dinov2_g_lda_within_macro_auc,
            "cross_script_macro_auc": dinov2_g_lda_cross_macro_auc,
        },
    ]
)

dinov2_g_summary_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitg14_reg_summary.csv"
)

dinov2_g_condition_path = (
    BENCHMARK_REPORT_DIR
    / "dinov2_vitg14_reg_condition_results.csv"
)

dinov2_g_summary_df.to_csv(
    dinov2_g_summary_path,
    index=False,
)

dinov2_g_condition_save_df = pd.concat(
    [
        dinov2_g_condition_df.assign(
            model="dinov2_vitg14_reg_frozen"
        ),
        dinov2_g_lda_condition_df.assign(
            model="dinov2_vitg14_reg_writer_lda"
        ),
    ],
    ignore_index=True,
)

dinov2_g_condition_save_df.to_csv(
    dinov2_g_condition_path,
    index=False,
)

print(
    "Embedding file:",
    dinov2_g_embedding_path,
)

print(
    "Embedding file size (MB):",
    dinov2_g_embedding_path.stat().st_size
    / 1024**2,
)

print(
    dinov2_g_summary_df
    .round(6)
    .to_string(
        index=False
    )
)

Embedding file: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\dinov2_vitg14_reg_embeddings.npz
Embedding file size (MB): 7.044621467590332
                       model  overall_auc      eer  within_script_macro_auc  cross_script_macro_auc
    dinov2_vitg14_reg_frozen     0.582887 0.450487                 0.731551                0.589826
dinov2_vitg14_reg_writer_lda     0.865667 0.214286                 0.941999                0.827771


In [49]:
giant_rows_df = pd.DataFrame(
    [
        {
            "model": "dinov2_vitg14_reg",
            "adaptation": "frozen",
            "overall_auc": dinov2_g_overall_auc,
            "eer": dinov2_g_overall_eer,
            "cross_script_macro_auc": dinov2_g_cross_macro_auc,
        },
        {
            "model": "dinov2_vitg14_reg",
            "adaptation": "writer_lda",
            "overall_auc": dinov2_g_lda_auc,
            "eer": dinov2_g_lda_eer,
            "cross_script_macro_auc": dinov2_g_lda_cross_macro_auc,
        },
    ]
)

benchmark_summary_df = pd.concat(
    [
        benchmark_summary_df[
            benchmark_summary_df[
                "model"
            ] != "dinov2_vitg14_reg"
        ],
        giant_rows_df,
    ],
    ignore_index=True,
)

benchmark_summary_df = (
    benchmark_summary_df
    .sort_values(
        "overall_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

benchmark_summary_df.to_csv(
    benchmark_summary_path,
    index=False,
)

print(
    benchmark_summary_df
    .round(6)
    .to_string(
        index=False
    )
)

print()
print(
    "Best overall model:",
    benchmark_summary_df.iloc[0][
        "model"
    ],
)

print(
    "Best overall AUC:",
    benchmark_summary_df.iloc[0][
        "overall_auc"
    ],
)

print(
    "Saved:",
    benchmark_summary_path,
)

                     model             adaptation  overall_auc      eer  cross_script_macro_auc
         dinov2_vitg14_reg             writer_lda     0.865667 0.214286                0.827771
         dinov2_vitl14_reg             writer_lda     0.827452 0.252976                0.791508
batch_alt_adversarial_mean task_specific_training     0.777343      NaN                0.746086
         efficientnet_v2_l             writer_lda     0.766926 0.309524                0.725764
         vit_b_16_swag_e2e             writer_lda     0.703563 0.350000                0.655685
         vit_h_14_swag_e2e             writer_lda     0.687981 0.369048                0.629408
         efficientnet_v2_l                 frozen     0.673399 0.375000                0.646618
         vit_b_16_swag_e2e                 frozen     0.671342 0.383929                0.647814
         dinov2_vitg14_reg                 frozen     0.582887 0.450487                0.589826
         dinov2_vitl14_reg              

In [50]:
batch_eer_columns = [
    column
    for column in multiseed_writer_df.columns
    if (
        "batch" in column.lower()
        and "eer" in column.lower()
    )
]

if len(batch_eer_columns) > 0:
    current_batch_mean_eer = float(
        multiseed_writer_df[
            batch_eer_columns[0]
        ].mean()
    )

    benchmark_summary_df.loc[
        benchmark_summary_df[
            "model"
        ] == "batch_alt_adversarial_mean",
        "eer",
    ] = current_batch_mean_eer

benchmark_summary_df = (
    benchmark_summary_df
    .sort_values(
        "overall_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

benchmark_summary_df.to_csv(
    benchmark_summary_path,
    index=False,
)

print(
    "Batch EER columns:",
    batch_eer_columns,
)

print()

print(
    benchmark_summary_df
    .round(6)
    .to_string(
        index=False
    )
)

Batch EER columns: ['batch_alt_eer']

                     model             adaptation  overall_auc      eer  cross_script_macro_auc
         dinov2_vitg14_reg             writer_lda     0.865667 0.214286                0.827771
         dinov2_vitl14_reg             writer_lda     0.827452 0.252976                0.791508
batch_alt_adversarial_mean task_specific_training     0.777343 0.292749                0.746086
         efficientnet_v2_l             writer_lda     0.766926 0.309524                0.725764
         vit_b_16_swag_e2e             writer_lda     0.703563 0.350000                0.655685
         vit_h_14_swag_e2e             writer_lda     0.687981 0.369048                0.629408
         efficientnet_v2_l                 frozen     0.673399 0.375000                0.646618
         vit_b_16_swag_e2e                 frozen     0.671342 0.383929                0.647814
         dinov2_vitg14_reg                 frozen     0.582887 0.450487                0.589826
  

In [51]:
benchmark_condition_df = pd.concat(
    [
        vit_condition_df.assign(
            model="vit_b_16_swag_e2e",
            adaptation="frozen",
        ),
        lda_condition_df.assign(
            model="vit_b_16_swag_e2e",
            adaptation="writer_lda",
        ),
        efficientnet_condition_df.assign(
            model="efficientnet_v2_l",
            adaptation="frozen",
        ),
        efficientnet_lda_condition_df.assign(
            model="efficientnet_v2_l",
            adaptation="writer_lda",
        ),
        dinov2_condition_df.assign(
            model="dinov2_vitl14_reg",
            adaptation="frozen",
        ),
        dinov2_lda_condition_df.assign(
            model="dinov2_vitl14_reg",
            adaptation="writer_lda",
        ),
        vit_h_condition_df.assign(
            model="vit_h_14_swag_e2e",
            adaptation="frozen",
        ),
        vit_h_lda_condition_df.assign(
            model="vit_h_14_swag_e2e",
            adaptation="writer_lda",
        ),
        dinov2_g_condition_df.assign(
            model="dinov2_vitg14_reg",
            adaptation="frozen",
        ),
        dinov2_g_lda_condition_df.assign(
            model="dinov2_vitg14_reg",
            adaptation="writer_lda",
        ),
    ],
    ignore_index=True,
)

benchmark_condition_path = (
    BENCHMARK_REPORT_DIR
    / "modern_baseline_condition_results.csv"
)

benchmark_condition_df.to_csv(
    benchmark_condition_path,
    index=False,
)

writer_lda_gain_df = pd.DataFrame(
    [
        {
            "model": "vit_b_16_swag_e2e",
            "overall_auc_gain": (
                lda_overall_auc
                - vit_overall_auc
            ),
            "cross_script_auc_gain": (
                lda_cross_macro_auc
                - vit_cross_macro_auc
            ),
        },
        {
            "model": "efficientnet_v2_l",
            "overall_auc_gain": (
                efficientnet_lda_auc
                - efficientnet_overall_auc
            ),
            "cross_script_auc_gain": (
                efficientnet_lda_cross_macro_auc
                - efficientnet_cross_macro_auc
            ),
        },
        {
            "model": "dinov2_vitl14_reg",
            "overall_auc_gain": (
                dinov2_lda_auc
                - dinov2_overall_auc
            ),
            "cross_script_auc_gain": (
                dinov2_lda_cross_macro_auc
                - dinov2_cross_macro_auc
            ),
        },
        {
            "model": "vit_h_14_swag_e2e",
            "overall_auc_gain": (
                vit_h_lda_auc
                - vit_h_overall_auc
            ),
            "cross_script_auc_gain": (
                vit_h_lda_cross_macro_auc
                - vit_h_cross_macro_auc
            ),
        },
        {
            "model": "dinov2_vitg14_reg",
            "overall_auc_gain": (
                dinov2_g_lda_auc
                - dinov2_g_overall_auc
            ),
            "cross_script_auc_gain": (
                dinov2_g_lda_cross_macro_auc
                - dinov2_g_cross_macro_auc
            ),
        },
    ]
)

writer_lda_gain_path = (
    BENCHMARK_REPORT_DIR
    / "writer_lda_adaptation_gains.csv"
)

writer_lda_gain_df.to_csv(
    writer_lda_gain_path,
    index=False,
)

print(
    "Condition report:",
    benchmark_condition_path,
)

print(
    "LDA gain report:",
    writer_lda_gain_path,
)

print()

print(
    writer_lda_gain_df
    .sort_values(
        "overall_auc_gain",
        ascending=False,
    )
    .round(6)
    .to_string(
        index=False
    )
)

Condition report: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\modern_baseline_condition_results.csv
LDA gain report: C:\Users\com\Documents\handwriting-cross-script-research\reports\modern_baseline_benchmarking\writer_lda_adaptation_gains.csv

            model  overall_auc_gain  cross_script_auc_gain
dinov2_vitg14_reg          0.282780               0.237945
dinov2_vitl14_reg          0.247577               0.205293
vit_h_14_swag_e2e          0.114809               0.038010
efficientnet_v2_l          0.093527               0.079145
vit_b_16_swag_e2e          0.032220               0.007871


# Final Summary — Modern Baseline Benchmarking

## Research Question

Can stronger modern and foundation visual representations outperform the current batch-alternating script-adversarial writer verifier under the exact same writer-disjoint QUWI verification protocol?

## Protocol

All models were evaluated using the same fixed protocol:

- 226 development writers / 904 development images
- 56 validation writers / 224 validation images
- 18,816 fixed validation verification pairs
- Cosine similarity on L2-normalized embeddings
- Overall AUC and EER
- Six page-pair conditions
- Within-script and cross-script macro AUC
- Writer-LDA was fitted only on development writers
- The official test split was not used for model selection

## Main Results

The strongest result was obtained by DINOv2 ViT-g/14-Reg with development-writer LDA adaptation:

- Overall AUC: 0.865667
- EER: 21.43%
- Cross-script macro AUC: 0.827771
- Within-script macro AUC: 0.941999

DINOv2 ViT-L/14-Reg with writer-LDA was the second strongest model:

- Overall AUC: 0.827452
- EER: 25.30%
- Cross-script macro AUC: 0.791508

The existing batch-alternating script-adversarial ResNet18 verifier achieved:

- Mean overall AUC: 0.777343
- Mean EER: 29.27%
- Mean cross-script macro AUC: 0.746086

Therefore, the existing compact verifier is no longer the strongest verifier under the controlled benchmark.

## Foundation Representation Finding

Raw frozen foundation embeddings were not sufficient for this task.

Frozen DINOv2-L and DINOv2-G achieved only approximately 0.58 overall AUC using direct cosine similarity.

However, development-only writer-LDA produced very large improvements:

- DINOv2-L overall AUC gain: +0.247577
- DINOv2-L cross-script gain: +0.205293
- DINOv2-G overall AUC gain: +0.282780
- DINOv2-G cross-script gain: +0.237945

This indicates that strong writer-discriminative information exists in the foundation representations, but it is not naturally aligned with the raw cosine verification space.

## Model Scale Finding

Model size alone did not determine verification quality.

The 632M-parameter ViT-H/14 SWAG model remained substantially weaker than the approximately 11M-parameter task-specific verifier after writer-LDA adaptation.

In contrast, scaling within the DINOv2 family from ViT-L to ViT-G improved the adapted model:

- Overall AUC: 0.827452 -> 0.865667
- Cross-script macro AUC: 0.791508 -> 0.827771

Therefore, representation quality, pretraining strategy, and task-specific adaptation appear more important than parameter count alone.

## Scientific Interpretation

This benchmark changes the role of the earlier batch-alternating adversarial model.

Its verification performance remains meaningful for a compact model, and its previous script-reliance findings remain scientifically relevant. However, it should not be presented as the strongest available verifier.

Future reliability and uncertainty experiments should therefore be evaluated against strong foundation-model baselines rather than only against the earlier compact verifier.

A separate future direction is to investigate whether a compact task-specific verifier can close the performance gap to DINOv2-based models while retaining its much smaller computational footprint.

## Conclusion

Modern foundation representations can substantially outperform the existing compact verifier when combined with development-only writer-discriminative adaptation.

The strongest controlled validation result in this notebook is DINOv2 ViT-g/14-Reg + writer-LDA with an overall AUC of 0.865667 and a cross-script macro AUC of 0.827771.

These results establish a substantially stronger benchmark for the next stage of the research, but they do not constitute a claim of global state-of-the-art performance.